# September on the ice
Choose two emissions pathways and compare projected Arctic sea-ice area in the same September. Each year is a single model result, so ice can grow or shrink between years. The bears illustrate scale changes in their surroundings; their number and locations are fixed.

Both the bear scenes and temperature data use the same CMIP6 model/member and pathways: CNRM-CM6-1, r1i1p1f2, SSP1-2.6 / SSP2-4.5 / SSP5-8.5. For a consistent region, this notebook derives sea-ice area from gridded sea-ice concentration north of 66.5°N, instead of using the original `siarean` Northern Hemisphere total. Temperature is also retained north of 66.5°N.

Run all cells with **dsc106-hurricane**. Data stores are from the [CMIP6 public archive](https://storage.googleapis.com/cmip6/pangeo-cmip6.csv).

In [1]:
from pathlib import Path
from urllib.request import urlopen
import hashlib
import html

import gcsfs
import pandas as pd
import xarray as xr
from IPython.display import HTML, display

cache = Path('arctic_cache')
cache.mkdir(exist_ok=True)
paths = ['ssp126', 'ssp245', 'ssp585']

In [2]:
# Derive Arctic-only sea-ice area from gridded concentration.
# The old ice.csv is the original Northern Hemisphere siarean series; it is
# left untouched as reference. This creates a separate Arctic-only cache file.
ice_file = cache / 'ice_arctic_north_of_66_5.csv'
gcs = gcsfs.GCSFileSystem(token='anon')
model = 'CNRM-CM6-1'
member = 'r1i1p1f2'
grid = 'gn'

# Discover exact store paths and versions instead of guessing them.
catalog_url = 'https://storage.googleapis.com/cmip6/pangeo-cmip6.csv'
cmip_catalog = pd.read_csv(catalog_url)

def latest_store(rows):
    if rows.empty:
        return None
    rows = rows.copy()
    rows['version_num'] = pd.to_numeric(rows['version'], errors='coerce')
    return rows.sort_values(['version_num', 'zstore'], kind='stable').iloc[-1].zstore

area_rows = cmip_catalog[
    (cmip_catalog.source_id == model)
    & (cmip_catalog.variable_id == 'areacello')
    & (cmip_catalog.grid_label == grid)
]
if 'table_id' in area_rows.columns:
    area_rows = area_rows[area_rows.table_id.isin(['Ofx', 'fx'])]
area_store = latest_store(area_rows)
if area_store is None:
    raise ValueError(f'No areacello store found for {model} on grid {grid}.')
print('Cell-area store:', area_store)

if not ice_file.exists():
    with xr.open_zarr(gcs.get_mapper(area_store), consolidated=False) as area_ds:
        cell_area = area_ds['areacello'].load()

    frames = []
    for scenario in paths:
        ice_rows = cmip_catalog[
            (cmip_catalog.source_id == model)
            & (cmip_catalog.member_id == member)
            & (cmip_catalog.experiment_id == scenario)
            & (cmip_catalog.table_id == 'SImon')
            & (cmip_catalog.variable_id == 'siconc')
            & (cmip_catalog.grid_label == grid)
        ]
        ice_store = latest_store(ice_rows)
        if ice_store is None:
            raise ValueError(
                f'No siconc store found for {model}/{member}/{scenario} on {grid}.'
            )
        print(f'{scenario} sea-ice store:', ice_store)
        with xr.open_zarr(gcs.get_mapper(ice_store), consolidated=False) as ds:
            concentration = ds['siconc']
            lat_name = 'lat' if 'lat' in concentration.coords else 'latitude'
            spatial_dims = [dim for dim in concentration.dims if dim != 'time']
            concentration = concentration.where(
                concentration[lat_name] >= 66.5, drop=True
            )
            regional_area = cell_area.where(
                cell_area[lat_name] >= 66.5, drop=True
            )
            units = str(concentration.attrs.get('units', '1')).lower()
            fraction = concentration / 100 if units in {'%', 'percent', 'percentage'} else concentration
            monthly_area_million_km2 = (
                (fraction * regional_area).sum(spatial_dims, skipna=True) / 1e12
            ).compute()
            frames.append(pd.DataFrame({
                'scenario': scenario,
                'year': monthly_area_million_km2.time.dt.year.values,
                'month': monthly_area_million_km2.time.dt.month.values,
                'area': monthly_area_million_km2.values,
            }))

    source_ice = pd.concat(frames, ignore_index=True)
    source_ice = source_ice[source_ice.year >= 2015].copy()
    assert set(source_ice.scenario) == set(paths)
    assert not source_ice.duplicated(['scenario', 'year', 'month']).any()
    assert source_ice.area.between(0, 30).all()
    temporary = ice_file.with_suffix('.tmp')
    source_ice.to_csv(temporary, index=False)
    temporary.replace(ice_file)

ice = pd.read_csv(ice_file)


Cell-area store: gs://cmip6/CMIP6/CMIP/CNRM-CERFACS/CNRM-CM6-1/historical/r29i1p1f2/Ofx/areacello/gn/v20200529/
ssp126 sea-ice store: gs://cmip6/CMIP6/ScenarioMIP/CNRM-CERFACS/CNRM-CM6-1/ssp126/r1i1p1f2/SImon/siconc/gn/v20190219/
ssp245 sea-ice store: gs://cmip6/CMIP6/ScenarioMIP/CNRM-CERFACS/CNRM-CM6-1/ssp245/r1i1p1f2/SImon/siconc/gn/v20190219/
ssp585 sea-ice store: gs://cmip6/CMIP6/ScenarioMIP/CNRM-CERFACS/CNRM-CM6-1/ssp585/r1i1p1f2/SImon/siconc/gn/v20190219/


In [3]:
september = ice.loc[ice.month == 9, ['scenario', 'year', 'area']]
assert len(september) == 3 * 86 and not september.duplicated(['scenario', 'year']).any()

### Matching monthly Arctic temperature data

The bear chart now uses sea-ice area calculated from `siconc × areacello`, summed over ocean grid cells north of 66.5°N. The temperature dataset uses the same model/member, scenarios, and latitude boundary. Its `tas` values remain spatially gridded; this cell does not average them. The temperature and sea-ice grids differ because they are different model components, but both are restricted to the same geographic region.


In [4]:
model = 'CNRM-CM6-1'
member = 'r1i1p1f2'
grid = 'gr'
version = 'v20190219'
base = ('gs://cmip6/CMIP6/ScenarioMIP/CNRM-CERFACS/'
        f'{model}')

tas_fields = []
for scenario in paths:
    store = (f'{base}/{scenario}/{member}/Amon/tas/{grid}/{version}/')
    with xr.open_zarr(gcs.get_mapper(store), consolidated=False) as ds:
        tas = ds['tas']
        lat_name = 'lat' if 'lat' in tas.coords else 'latitude'
        # Keep all time steps from the bear visualization period onward.
        tas = tas.where(tas.time.dt.year >= 2015, drop=True)
        # Keep the same Arctic latitude boundary as the sea-ice calculation.
        tas = tas.where(tas[lat_name] >= 66.5, drop=True).load()
        tas_fields.append(tas.expand_dims(scenario=[scenario]))

# Check that pathway arrays have identical coordinates before combining them.
temperature = xr.concat(tas_fields, dim='scenario', join='exact')
temperature.name = 'tas'
temperature.attrs.update({
    'long_name': 'Near-surface air temperature',
    'units': 'K',
    'region': 'grid cells north of 66.5 degrees north',
    'source_id': model,
    'member_id': member,
    'grid_label': grid,
    'version': version,
})

# siarean is a monthly Northern Hemisphere total; document its dimensions
# beside the spatial temperature field to make the different structures clear.
print('Sea-ice area data columns:', list(ice.columns))
print('Temperature field dimensions:', dict(temperature.sizes))
print('Temperature pathways:', list(temperature.scenario.values))


Sea-ice area data columns: ['scenario', 'year', 'month', 'area']
Temperature field dimensions: {'scenario': 3, 'time': 1032, 'lat': 17, 'lon': 256}
Temperature pathways: ['ssp126', 'ssp245', 'ssp585']


Save the full monthly temperature grid and Arctic-only sea-ice series. The bear scenes and any later line plot can now use the same region and scenarios. The temperature values remain gridded; sea ice is summarized as regional area per month.

In [5]:
tas_file = cache / 'arctic_tas_monthly_CNRM-CM6-1_r1i1p1f2.nc'
temperature.to_dataset().to_netcdf(tas_file)
print(f'Saved full Arctic temperature data to {tas_file}')


Saved full Arctic temperature data to arctic_cache/arctic_tas_monthly_CNRM-CM6-1_r1i1p1f2.nc


In [8]:
# Find a template that embeds the generated data and scripts inline.
template_candidates = [
    Path('data/ice-floes.html'),
    Path('ice-floes.html'),
    Path('archive/arctic-notebook-marta/ice-floes.html'),
]
page_template = next((candidate for candidate in template_candidates
    if candidate.is_file() and 'ICE_DATA' in candidate.read_text()
    and '/* LIBRARY */' in candidate.read_text()
    and '/* DRAW */' in candidate.read_text()), None)
if page_template is None:
    raise FileNotFoundError(
        'Could not find an ice-floes.html template containing ICE_DATA, '
        '/* LIBRARY */, and /* DRAW */. Checked data/, the project root, and archive/arctic-notebook-marta/.'
    )
script_file = page_template.with_name('ice-floes.js')
if not script_file.is_file():
    raise FileNotFoundError(f'Could not find ice-floes.js next to {page_template}')

# Keep the pinned D3 copy already downloaded into the notebook cache.
d3_file = cache / 'd3.min.js'
if not d3_file.exists():
    with urlopen('https://cdn.jsdelivr.net/npm/d3@7.9.0/dist/d3.min.js', timeout=60) as response:
        script = response.read()
    assert hashlib.sha256(script).hexdigest() == 'f2094bbf6141b359722c4fe454eb6c4b0f0e42cc10cc7af921fc158fceb86539'
    temporary = d3_file.with_suffix('.tmp')
    temporary.write_bytes(script)
    temporary.replace(d3_file)

page = (page_template.read_text()
        .replace('ICE_DATA', september.to_json(orient='records'))
        .replace('/* LIBRARY */', d3_file.read_text())
        .replace('/* DRAW */', script_file.read_text()))
print(f'Using visualization template: {page_template}')
frame = '<iframe title="September on the ice" style="width:100%;height:670px;border:0" srcdoc="{}"></iframe>'
display(HTML(frame.format(html.escape(page, quote=True))))


/Users/evgenykrylov/miniforge3/envs/dsc106-hurricane/lib/python3.12/site-packages/IPython/core/display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
